# Homework 2 反向传播与自动求导

Author: Lianjie (Marc) Shi  
请继续使用 `torch` 的完善上周的模拟实验：


4. （iv）请利用 torch 中的自动求导功能，实现小批量梯度下降算法计算最小二乘估计
- MGD的更新应该是一个两层循环，首先将全样本切分为 $M$ 个mini－batch，利用每一个mini－batch的样本来进行前向传播，计算loss，随后进行反向传播，按照梯度更新估计量；
- 外层循环是 $T$ 个Epoch；
- 算法最终的收敛可以用最大迭代次数或者是否收敛来判断．

In [1]:
import torch
import numpy as np

In [ ]:
torch.manual_seed(42)

n = 1000
p = 10
batch_size = 50
alpha = 0.001    # 学习率
T = 200          # epoch轮数, 和hw1保持一致
n_repeat = 100

beta0 = torch.rand(p, 1)  

X = torch.randn(n, p)          # X: (n,p), 每行X_i ~ N(0, I_p)
eps = torch.randn(n, 1) * 0.4  # epsilon: N(0, 0.16), 标准差0.4
Y = X @ beta0 + eps

beta_mgd_autograd = torch.zeros(p, 1, requires_grad=True)  

#用最大迭代次数进行迭代
for epoch in range(T):
    perm = torch.randperm(n) 
    X_shuffle = X[perm, :]
    Y_shuffle = Y[perm, :]
    
    for start in range(0, n, batch_size):
        end = start + batch_size
        Xb = X_shuffle[start:end, :]
        Yb = Y_shuffle[start:end, :]
        
        Y_hat = Xb @ beta_mgd_autograd
        loss = torch.mean((Y_hat - Yb) ** 2)
        
        loss.backward()
    
        with torch.no_grad():
            beta_mgd_autograd -= alpha * beta_mgd_autograd.grad
        
        # 梯度清零
        beta_mgd_autograd.grad.zero_()

print("AutoGrad MGD estimate beta_mgd:\n", beta_mgd_autograd.detach())

AutoGrad MGD estimate beta_mgd:
 tensor([[0.8799],
        [0.9207],
        [0.3944],
        [0.9509],
        [0.3771],
        [0.5959],
        [0.2478],
        [0.7739],
        [0.9572],
        [0.1300]])


5．重复步骤 $2-4$ ，进行 100 次重复实验，和上周得到的手动计算版本做对比．

In [ ]:
beta_autograd_list = []

for rep in range(n_repeat):
    # 每次重新生成数据
    X = torch.randn(n, p)
    eps = torch.randn(n, 1) * 0.4
    Y = X @ beta0 + eps
    
    # 自动求导MGD
    beta = torch.zeros(p, 1, requires_grad=True)
    for epoch in range(T):
        perm = torch.randperm(n)
        X_shuffle = X[perm, :]
        Y_shuffle = Y[perm, :]
        for start in range(0, n, batch_size):
            end = start + batch_size
            Xb = X_shuffle[start:end, :]
            Yb = Y_shuffle[start:end, :]
            Y_hat = Xb @ beta
            loss = torch.mean((Y_hat - Yb) ** 2)
            loss.backward()
            with torch.no_grad():
                beta -= alpha * beta.grad
            beta.grad.zero_()
    
    beta_autograd_list.append(beta.detach().clone())

beta_autograd_all = torch.stack(beta_autograd_list)
norm2_autograd = torch.sum((beta_autograd_all - beta0) ** 2, dim=1)
rmse_autograd = torch.sqrt(torch.mean(norm2_autograd))
print(f"\nRMSE of AutoGrad MGD: {rmse_autograd.item():.6f}")


RMSE of AutoGrad MGD: 0.038966


从结果我们可以看到自动求导版本的 RMSE 与上周手动计算的 MGD 版本完全一致,为$0.038966$。这个验证了PyTorch自动求导的数学正确性。由此我们发现
- 手动求导需要预先推导梯度公式，每更换损失函数都要重新推导，容易出错；
- 自动求导通过前向传播时构建计算图，反向传播自动完成链式求导，代码更简洁，扩展性更强。

6. 自主搜索文献，了解对于深度学习训练来说，MGD算法应该如何动态调整学习率。选择一种方式进行上述实验，与常数学习率下的结果做对比．

In [8]:
beta_adam_list = []
for rep in range(n_repeat):
    X = torch.randn(n, p)
    eps = torch.randn(n, 1) * 0.4
    Y = X @ beta0 + eps
    
    beta_adam = torch.zeros(p, 1, requires_grad=True)
    optimizer = torch.optim.Adam([beta_adam], lr=alpha)
    
    for epoch in range(T):
        perm = torch.randperm(n)
        X_shuffle = X[perm, :]
        Y_shuffle = Y[perm, :]
        for start in range(0, n, batch_size):
            end = start + batch_size
            Xb = X_shuffle[start:end, :]
            Yb = Y_shuffle[start:end, :]
            Y_hat = Xb @ beta_adam
            loss = torch.mean((Y_hat - Yb) ** 2)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    beta_adam_list.append(beta_adam.detach().clone())

beta_adam_all = torch.stack(beta_adam_list)
norm2_adam = torch.sum((beta_adam_all - beta0)**2, dim=1)
rmse_adam = torch.sqrt(torch.mean(norm2_adam))
print(f"\nRMSE of Adam adaptive lr MGD: {rmse_adam.item():.6f}")



RMSE of Adam adaptive lr MGD: 0.038580


通过搜索文献，我采用了Adam算法进行动态自适应调整学习率，发现：

1. **收敛效果**：相同初始学习率、相同训练轮次下，Adam 自适应学习率的最终损失更低，参数估计 RMSE 更小，更接近真实回归系数 $\beta_0$。
2. **原理差异**：固定学习率 MGD 对所有参数使用统一更新步长；Adam 为每个参数独立调整步长 —— 梯度波动大的参数自动缩小步长，梯度平稳的参数自动放大步长，对小批量梯度的随机噪声鲁棒性更强。
参数更新规则：

$$
\theta_{t+1}=\theta_t-\alpha \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\epsilon}
$$
- 分母 $\sqrt{\hat{v}_t}$ 实现每个参数独立的动态缩放：梯度波动大的参数，分母大，有效步长自动缩小；梯度平稳的参数，分母小，有效步长自动放大。